# DMB baryon model observables with hmfast

This notebook integrates the **DMB** (Dark Matter + Baryon) halo model from GODMAX / Schneider–Giri into hmfast and plots:

1. Real-space $P_e(r)$ and $\rho_{\rm dmb}(r)$ vs GODMAX
2. tSZ $C_\ell^{yy}$ (DMB vs GNFW vs B12)
3. Matter $P(k)$ suppression $P_{\rm dmb}/P_{\rm nfw}$
4. Lensing $C_\ell^{\kappa\kappa}$ (DMB vs NFW)
5. Shear–y $C_\ell^{\kappa y}$ and configuration-space `gty`, $\xi_\pm$
6. kSZ with DMB gas vs Schneider `BCMDensityProfile` / B16

Activate the project env first:

```bash
source /scratch/scratch-lxu/venv/cmbagent_env/bin/activate
```


In [ ]:
import os
USE_GPU = False  # DMB profile evaluation is CPU-friendly for this tutorial
if not USE_GPU:
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)
print("JAX devices:", jax.devices())

from hmfast.cosmology import Cosmology
from hmfast.halos import HaloModel
from hmfast.halos.profiles import (
    DMBPressureProfile, DMBMatterProfile, DMBGasDensityProfile,
    GNFWPressureProfile, B12PressureProfile, NFWMatterProfile,
    B16DensityProfile, BCMDensityProfile,
)
from hmfast.halos.profiles.dmb import dmb_halo_quantities, _DEFAULTS
from hmfast.halos.cl_to_xi import (
    cl_ky_to_gty, cl_kk_to_xip, cl_kk_to_xim, theta_to_arcmin, interp_xi_theta,
)
from hmfast.tracers import tSZTracer, GalaxyLensingTracer, CMBLensingTracer, kSZTracer


In [ ]:
cosmo = Cosmology(emulator_set="lcdm:v1")
hm = HaloModel(cosmology=cosmo)
cp = cosmo._cosmo_params()
h = float(cp["h"]); Ob0 = float(cp["Omega_b"]); Om0 = float(cp["Omega0_m"])
print(f"h={h:.3f}, Ob={Ob0:.4f}, Om={Om0:.4f}")

# Shared grids (keep modest for interactive runtime)
m = jnp.logspace(12.5, 15.0, 24) / h   # physical Msun (GODMAX-like Msun/h grid)
z = jnp.linspace(0.05, 1.2, 16)
ell = jnp.logspace(np.log10(30), np.log10(3000), 40)
k_arr = jnp.logspace(-2, 1, 40)


## 1. Profiles vs GODMAX

In [ ]:
# Match a single halo to GODMAX BCM_18_wP
M_h, z0, c0 = 1e14, 0.2, 5.0
m_phys = M_h / h
params = {**_DEFAULTS, "num_points_trapz_int": 48}
q = dmb_halo_quantities(m_phys, z0, c0, Ob0, Om0, h, params)

# Optional GODMAX overlay
godmax_ok = False
try:
    import sys
    sys.path.insert(0, os.path.abspath("../ref_packages/GODMAX/src"))
    from get_BCMP_profile_jit import BCM_18_wP
    sim = dict(
        cosmo=dict(H0=100*h, Om0=Om0, Ob0=Ob0, sigma8=0.81, ns=0.96, w0=-1.0),
        theta_ej_0=4.0, theta_co_0=0.1, log10_Mc0=14.83, mu_beta=0.21,
        eta_star=0.3, eta_cga=0.6, A_starcga=0.09, log10_M1_starcga=11.4,
        alpha_nt=0.18, beta_nt=0.5, n_nt=0.3, gamma_rhogas=2., delta_rhogas=7.,
        nfw_trunc=True, epsilon_rt=4.0,
    )
    halo = dict(rmin=5e-3, rmax=3., nr=48, z_array=[z0],
                lg10_Mmin=np.log10(M_h), lg10_Mmax=np.log10(M_h), nM=1,
                cmin=c0, cmax=c0, nc=1)
    bcmp = BCM_18_wP(sim, halo, num_points_trapz_int=48)
    r_g = np.asarray(bcmp.r_array) / h
    pe_g = np.asarray(bcmp.Pe_mat_physical[:,0,0,0]) * 1000.0  # keV -> eV
    rho_g = np.asarray(bcmp.rho_dmb_mat[:,0,0,0]) * h**2
    godmax_ok = True
    print("GODMAX comparison loaded")
except Exception as e:
    print("GODMAX overlay skipped:", e)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
x = np.asarray(q["r_comoving"]) / float(q["r200c_comoving"])
ax[0].loglog(x, np.asarray(q["Pe"]), "k-", label="hmfast DMB")
if godmax_ok:
    ax[0].loglog(r_g/float(q["r200c_comoving"]), pe_g, "C0--", label="GODMAX")
ax[0].set_xlabel(r"$r / r_{200c}$"); ax[0].set_ylabel(r"$P_e$ [eV/cm$^3$]")
ax[0].legend(); ax[0].set_title("Electron pressure")

ax[1].loglog(x, np.asarray(q["rho_dmb"]), "k-", label="hmfast DMB")
if godmax_ok:
    ax[1].loglog(r_g/float(q["r200c_comoving"]), rho_g, "C0--", label="GODMAX")
ax[1].set_xlabel(r"$r / r_{200c}$"); ax[1].set_ylabel(r"$\rho_{\rm dmb}$ [M$_\odot$/Mpc$^3$]")
ax[1].legend(); ax[1].set_title("Total DMB density")
plt.tight_layout(); plt.show()


## 2. tSZ $C_\ell^{yy}$: DMB vs GNFW vs B12

In [ ]:
tsz_dmb = tSZTracer(profile=DMBPressureProfile(num_points_trapz_int=32))
tsz_gnfw = tSZTracer(profile=GNFWPressureProfile(B=1.4))
tsz_b12 = tSZTracer(profile=B12PressureProfile())

def cl_yy(tracer):
    return np.asarray(hm.cl_1h(tracer, None, ell, m, z) + hm.cl_2h(tracer, None, ell, m, z))

cl_dmb = cl_yy(tsz_dmb)
cl_gnfw = cl_yy(tsz_gnfw)
cl_b12 = cl_yy(tsz_b12)

plt.figure(figsize=(6,4))
plt.loglog(ell, cl_dmb, label="DMB")
plt.loglog(ell, cl_gnfw, label="GNFW (B=1.4)")
plt.loglog(ell, cl_b12, label="Battaglia12")
plt.xlabel(r"$\ell$"); plt.ylabel(r"$C_\ell^{yy}$")
plt.legend(); plt.title("tSZ angular power spectrum")
plt.tight_layout(); plt.show()


## 3. Matter $P(k)$ suppression $P_{\rm dmb}/P_{\rm nfw}$

Paper-style diagnostic (GODMAX / To–Dalal): stronger feedback (larger $\theta_{ej}$) suppresses small-scale power.

In [ ]:
lens_nfw = CMBLensingTracer(profile=NFWMatterProfile())  # uses matter u_k via profile
# Use matter profiles through CMB lensing tracer for pk_* API, or call pk with matter-like tracers.
# HaloModel.pk expects Tracer objects; CMBLensingTracer wraps MatterProfile.

def pmm(tracer, z_idx=3):
    pk = hm.pk_1h(tracer, None, k_arr, m, z) + hm.pk_2h(tracer, None, k_arr, m, z)
    return np.asarray(pk)[:, z_idx]  # shape depends on API

# Check pk shape
_pk = hm.pk_1h(lens_nfw, None, k_arr, m, z)
print("pk_1h shape", np.shape(_pk))

lens_dmb = CMBLensingTracer(profile=DMBMatterProfile(num_points_trapz_int=32))
pk_nfw = np.asarray(hm.pk_1h(lens_nfw, None, k_arr, m, z) + hm.pk_2h(lens_nfw, None, k_arr, m, z))
pk_dmb = np.asarray(hm.pk_1h(lens_dmb, None, k_arr, m, z) + hm.pk_2h(lens_dmb, None, k_arr, m, z))

# Handle (Nk, Nz) or (Nk,)
if pk_nfw.ndim == 2:
    zi = int(np.argmin(np.abs(np.asarray(z) - 0.3)))
    ratio = pk_dmb[:, zi] / np.maximum(pk_nfw[:, zi], 1e-30)
else:
    ratio = pk_dmb / np.maximum(pk_nfw, 1e-30)

# Parameter variation in theta_ej
plt.figure(figsize=(6,4))
for tej in [2.0, 4.0, 8.0]:
    tr = CMBLensingTracer(profile=DMBMatterProfile(theta_ej_0=tej, num_points_trapz_int=32))
    pk = np.asarray(hm.pk_1h(tr, None, k_arr, m, z) + hm.pk_2h(tr, None, k_arr, m, z))
    if pk.ndim == 2:
        r = pk[:, zi] / np.maximum(pk_nfw[:, zi], 1e-30)
    else:
        r = pk / np.maximum(pk_nfw, 1e-30)
    plt.semilogx(k_arr, r, label=fr"$\theta_{{ej}}={tej}$")
plt.axhline(1.0, color="k", ls=":")
plt.xlabel(r"$k$ [Mpc$^{-1}$]"); plt.ylabel(r"$P_{\rm dmb}/P_{\rm nfw}$")
plt.legend(); plt.title(r"Matter power suppression at $z\approx 0.3$")
plt.tight_layout(); plt.show()


## 4. Lensing $C_\ell^{\kappa\kappa}$ (galaxy + CMB)

In [ ]:
gal_nfw = GalaxyLensingTracer(profile=NFWMatterProfile())
gal_dmb = GalaxyLensingTracer(profile=DMBMatterProfile(num_points_trapz_int=32))
cmb_nfw = CMBLensingTracer(profile=NFWMatterProfile())
cmb_dmb = CMBLensingTracer(profile=DMBMatterProfile(num_points_trapz_int=32))

def cl_auto(tr):
    return np.asarray(hm.cl_1h(tr, None, ell, m, z) + hm.cl_2h(tr, None, ell, m, z))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].loglog(ell, cl_auto(gal_nfw), label="NFW")
ax[0].loglog(ell, cl_auto(gal_dmb), label="DMB")
ax[0].set_xlabel(r"$\ell$"); ax[0].set_ylabel(r"$C_\ell^{\kappa\kappa}$ (galaxy)")
ax[0].legend(); ax[0].set_title("Galaxy lensing")

ax[1].loglog(ell, cl_auto(cmb_nfw), label="NFW")
ax[1].loglog(ell, cl_auto(cmb_dmb), label="DMB")
ax[1].set_xlabel(r"$\ell$"); ax[1].set_ylabel(r"$C_\ell^{\kappa\kappa}$ (CMB)")
ax[1].legend(); ax[1].set_title("CMB lensing")
plt.tight_layout(); plt.show()


## 5. Shear–y $C_\ell^{\kappa y}$, `gty`, and $\xi_\pm$

In [ ]:
# Cross: galaxy lensing (DMB matter) x tSZ (DMB pressure)
cl_ky = np.asarray(
    hm.cl_1h(gal_dmb, tsz_dmb, ell, m, z) + hm.cl_2h(gal_dmb, tsz_dmb, ell, m, z)
)
cl_kk = cl_auto(gal_dmb)

theta_gty, gty = cl_ky_to_gty(ell, cl_ky)
theta_xip, xip = cl_kk_to_xip(ell, cl_kk)
theta_xim, xim = cl_kk_to_xim(ell, cl_kk)
th_am = np.asarray(theta_to_arcmin(theta_gty))

# Vary theta_ej for gty (paper-style)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
ax[0].loglog(ell, np.abs(cl_ky), "k-")
ax[0].set_xlabel(r"$\ell$"); ax[0].set_ylabel(r"$|C_\ell^{\kappa y}|$")
ax[0].set_title("Shear–y power")

for tej, ls in zip([2.0, 4.0, 8.0], ["-", "--", ":"]):
    gal = GalaxyLensingTracer(profile=DMBMatterProfile(theta_ej_0=tej, num_points_trapz_int=32))
    tsz = tSZTracer(profile=DMBPressureProfile(theta_ej_0=tej, num_points_trapz_int=32))
    cky = np.asarray(hm.cl_1h(gal, tsz, ell, m, z) + hm.cl_2h(gal, tsz, ell, m, z))
    th, g = cl_ky_to_gty(ell, cky)
    ax[1].loglog(theta_to_arcmin(th), np.abs(g), ls=ls, label=fr"$\theta_{{ej}}={tej}$")
ax[1].set_xlabel(r"$\theta$ [arcmin]"); ax[1].set_ylabel(r"$|gty|$")
ax[1].legend(); ax[1].set_title("gty (Hankel of $C_\ell^{\kappa y}$)")

ax[2].loglog(theta_to_arcmin(theta_xip), np.abs(xip), label=r"$\xi_+$")
ax[2].loglog(theta_to_arcmin(theta_xim), np.abs(xim), label=r"$\xi_-$")
ax[2].set_xlabel(r"$\theta$ [arcmin]"); ax[2].set_ylabel(r"$|\xi_\pm|$")
ax[2].legend(); ax[2].set_title(r"Shear 2pt from $C_\ell^{\kappa\kappa}$")
plt.tight_layout(); plt.show()


## 6. kSZ: DMB gas vs BCM / B16 (existing hmfast tracers)

In [ ]:
ksz_dmb = kSZTracer(profile=DMBGasDensityProfile(num_points_trapz_int=32))
ksz_bcm = kSZTracer(profile=BCMDensityProfile())
ksz_b16 = kSZTracer(profile=B16DensityProfile())

def cl_ksz(tr):
    return np.asarray(hm.cl_1h(tr, None, ell, m, z) + hm.cl_2h(tr, None, ell, m, z))

plt.figure(figsize=(6,4))
plt.loglog(ell, cl_ksz(ksz_dmb), label="DMB gas")
plt.loglog(ell, cl_ksz(ksz_bcm), label="Schneider BCM")
plt.loglog(ell, cl_ksz(ksz_b16), label="Battaglia16")
plt.xlabel(r"$\ell$"); plt.ylabel(r"$C_\ell^{\rm kSZ}$")
plt.legend(); plt.title("kSZ angular power (existing hmfast tracer)")
plt.tight_layout(); plt.show()


## Summary

- **DMB profiles** live in `hmfast.halos.profiles` (`DMBPressureProfile`, `DMBMatterProfile`, `DMBGasDensityProfile`).
- They drop into existing tracers (`tSZTracer`, lensing, `kSZTracer`) without API changes.
- Config-space `gty` / $\xi_\pm$ use `hmfast.halos.cl_to_xi`.
- Quantitative profile agreement with GODMAX is covered by `tests/test_dmb_profiles.py`.
